In [5]:
# load packages
import numpy as np 
import scipy.io
from scipy.io   import  loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt #import matplotlib as plt
from scipy.optimize import curve_fit 
import seaborn as sns #import mat73
import pickle as pkl
from datetime import datetime
from itertools import chain
from scipy.optimize import curve_fit
from scipy.io import savemat

In [6]:
from pathlib import Path
MainDir = Path.home() / "Desktop" / "Visual_Plasticty_Pipeline" / "EEG RCA"

os.chdir(MainDir) # change old dir, to this dir
d = os.listdir(MainDir) # list files in dir
print(f'Avilable Files to choose from: {len(d)}')
print(f'Files on hand: {d}')
##############################################
FileN_f1 = d[2]#d[1] # choose one
FileN_f2  = d[1]# d[3]                         
file_path1 = os.path.join(MainDir, FileN_f1) # join paths and prep 2 load
print('Current WD:',file_path1) # does path exist ... ?
print('Does File #1 Exist?',os.path.exists(file_path1)) # yes or no

file_path2 = os.path.join(MainDir, FileN_f2) # join paths and prep 2 load
print('Current WD:',file_path2) # does path exist ... ?
print('Does File #2 Exist?',os.path.exists(file_path1)) # yes or no

Avilable Files to choose from: 3
Files on hand: ['.DS_Store', 'rcaResults_Sweep_ContrastSweeps_RightHemiF_F2.mat', 'rcaResults_Sweep_ContrastSweeps_LeftHemiF_F1.mat']
Current WD: /Users/patricia.naomi/Desktop/Visual_Plasticty_Pipeline/EEG RCA/rcaResults_Sweep_ContrastSweeps_LeftHemiF_F1.mat
Does File #1 Exist? True
Current WD: /Users/patricia.naomi/Desktop/Visual_Plasticty_Pipeline/EEG RCA/rcaResults_Sweep_ContrastSweeps_RightHemiF_F2.mat
Does File #2 Exist? True


In [7]:
# import rca signal data | F1 data and F2 data
df_f1 = scipy.io.loadmat(file_path1) # RCA F1 File
df_f2 = scipy.io.loadmat(file_path2)# RCA F2 File

print(int(len(df_f1)))

4


In [8]:
# sort data imported above
rca_f1 = df_f1['rcaResult']['projectedData'][0,0]
f1 = [rca_f1[x,0] for x in range(rca_f1.shape[0])] # entry per subject

rca_f2 = df_f2['rcaResult']['projectedData'][0,0]
f2 = [rca_f2[x,0] for x in range(rca_f2.shape[0])]

print(len(rca_f1))
SignalFiles = [f1, f2] # Hemifield data

65


In [9]:
# load subject names ...
SubNames = df_f1['rcaResult'][0,0][5]
FileName = [x[0][3:] for subjlist in SubNames for x in subjlist[0][2][0]] # rm 'nl-'
UniformFileNames = [str(s).replace('-', '_') for s in FileName]
print(UniformFileNames)

print(UniformFileNames[-1])
fixName = '345216_attnL_20230929_1732'
UniformFileNames[-1] = fixName
print(UniformFileNames[-1])

['2651_attnL_20231003_1500', '2651_attnR_20231006_0933', '2652_attnL_20231003_1635', '2652_attnR_20231011_1328', '2653_attnL_20231009_1015', '2653_attnR_20231013_0932', '2654_attnL_20231009_1131', '2654_attnR_20231016_1152', '2655_attnL_20231009_1303', '2655_attnR_20231016_0948', '2657_attnL_20231013_1508', '2657_attnR_20231020_1201', '2658_attnL_20231013_1639', '2658_attnR_20231020_1052', '2659_attnL_20231017_0940', '2659_attnR_20231018_1523', '2660_attnL_20231017_1102', '2661_attnL_20231018_1322', '2661_attnR_20231017_1358', '2663_attnR2_20231030_1659', '2663_attnR_20231019_1018', '2664_attnL_20231020_1452', '2664_attnR_20231019_1145', '2665_attnL_20231024_1329', '2665_attnR_20231019_1512', '2666_attnL_20231023_1612', '2666_attnR_20231019_1643', '2667_attnL_20231025_1522', '2667_attnR_20231023_0947', '2668_attnL_20231027_1251', '2668_attnR_20231023_1058', '2669_attnR_20231023_1241', '2669_attnL_20231024_1432', '2670_attnR_20231024_0941', '2670_attnL_20231201_1521', '2672_attnL_202310

In [10]:
NumFiles = int(len(FileName))
print(f'Number of Files to Sort: {NumFiles}')

Number of Files to Sort: 65


In [11]:
File_Split_CatchString = '_'
SubjectNum_Array = [x.split(File_Split_CatchString)[0] for x in UniformFileNames]
print(SubjectNum_Array)
SubjExpDates = [x.split(File_Split_CatchString)[2] for x in UniformFileNames]
print(SubjExpDates)
AttentionCond = [x.split(File_Split_CatchString)[1] for x in UniformFileNames] # 3 was 2 
print(AttentionCond)
print(np.unique(AttentionCond))

['2651', '2651', '2652', '2652', '2653', '2653', '2654', '2654', '2655', '2655', '2657', '2657', '2658', '2658', '2659', '2659', '2660', '2661', '2661', '2663', '2663', '2664', '2664', '2665', '2665', '2666', '2666', '2667', '2667', '2668', '2668', '2669', '2669', '2670', '2670', '2672', '2672', '2674', '2674', '2676', '2677', '2677', '2678', '2695', '2695', '2696', '2696', '2697', '2697', '2708', '2715', '2716', '2726', '2727', '2728', '2728', '2733', '2734', '2737', '345202', '345202', '345215', '345215', '345216', '345216']
['20231003', '20231006', '20231003', '20231011', '20231009', '20231013', '20231009', '20231016', '20231009', '20231016', '20231013', '20231020', '20231013', '20231020', '20231017', '20231018', '20231017', '20231018', '20231017', '20231030', '20231019', '20231020', '20231019', '20231024', '20231019', '20231023', '20231019', '20231025', '20231023', '20231027', '20231023', '20231023', '20231024', '20231024', '20231201', '20231030', '20231025', '20231026', '20231031'

In [12]:
'get unique id numbers to call data >;D'
NumSubs = int(len(np.unique(SubjectNum_Array)))
print(NumSubs)
SubjId, counts = np.unique(SubjectNum_Array, return_counts= True)
print(SubjId)
SubjBool = np.zeros((NumFiles))
for i in range(NumSubs): # NumSubs
    sID = str(SubjId[i]) # a number in array
    #print([x for x in SubjectNum_Array if x == sID])
    indicies = np.where(np.array(SubjectNum_Array) == sID)[0]
    # print(indicies)
    SubjBool[indicies] = int(i)

SessInds = np.array([int(item[1:]) for item in SubjExpDates]) # session array per subj :)

print(SubjBool) # unique subject names based on occurance

38
['2651' '2652' '2653' '2654' '2655' '2657' '2658' '2659' '2660' '2661'
 '2663' '2664' '2665' '2666' '2667' '2668' '2669' '2670' '2672' '2674'
 '2676' '2677' '2678' '2695' '2696' '2697' '2708' '2715' '2716' '2726'
 '2727' '2728' '2733' '2734' '2737' '345202' '345215' '345216']
[ 0.  0.  1.  1.  2.  2.  3.  3.  4.  4.  5.  5.  6.  6.  7.  7.  8.  9.
  9. 10. 10. 11. 11. 12. 12. 13. 13. 14. 14. 15. 15. 16. 16. 17. 17. 18.
 18. 19. 19. 20. 21. 21. 22. 23. 23. 24. 24. 25. 25. 26. 27. 28. 29. 30.
 31. 31. 32. 33. 34. 35. 35. 36. 36. 37. 37.]


In [13]:
# Get the unique numbers and counts of occurrences
unique_vals, counts = np.unique(SubjBool, return_counts=True)

# Generate the desired output
NumFilePerSubj = np.concatenate([np.arange(1, count + 1) for count in counts])
print(NumFilePerSubj)

[1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2
 1 2 1 1 2 1 1 2 1 2 1 2 1 1 1 1 1 1 2 1 1 1 1 2 1 2 1 2]


In [14]:
print(SubjectNum_Array)

['2651', '2651', '2652', '2652', '2653', '2653', '2654', '2654', '2655', '2655', '2657', '2657', '2658', '2658', '2659', '2659', '2660', '2661', '2661', '2663', '2663', '2664', '2664', '2665', '2665', '2666', '2666', '2667', '2667', '2668', '2668', '2669', '2669', '2670', '2670', '2672', '2672', '2674', '2674', '2676', '2677', '2677', '2678', '2695', '2695', '2696', '2696', '2697', '2697', '2708', '2715', '2716', '2726', '2727', '2728', '2728', '2733', '2734', '2737', '345202', '345202', '345215', '345215', '345216', '345216']


In [15]:
SortedSessionDates = np.zeros((NumSubs*2))

session_day_delay = np.zeros((NumSubs))
print(SortedSessionDates.shape)

from datetime import datetime

(76,)


In [16]:
# fix log 
# 4/16/25
# 5/7/25 whatever that fix was, was WRONG - Fixed: Session Dates correctly Day 1 = 1, Day 2  = 2 , No Data = 0
#  For this fix i just changed the default values for sessSorted to be filled with 1s and set missing / data day 2 to either 0 or 2 respectively
for nS in range(NumSubs): # NumSubs
    print(nS)
    sessSorted = np.full((1,2), 1)
    subjIn = SubjId[nS]
    # print(subjIn)
    datePos = np.where(SubjBool == nS)[0]
    np.array(datePos)
    # print(subjIn,datePos)
    if datePos.size != 2:
        print(f'WARNING {SubjId[nS]}-  is a single session..NOW FIXING')
        # this section adds a nan as missing data file [no 2nd session]
        # sessSorted[0,0] = 1 #datePos[0] was 0 on 5/6/25
        sessSorted[0,1] = 0
        s = nS * 2
        e = (nS+1) * 2
        # print(s,e)
        SortedSessionDates[s:e] = sessSorted
        day_delay = np.nan
        session_day_delay[nS] = day_delay
        #print(sessSorted)
    else:
        print(f'IMPORTING {SubjId[nS]} - Completed 2 sessions')# datePos
        date1 = int(SubjExpDates[datePos[0]])
        date2 = int(SubjExpDates[datePos[1]])
        #print(date1,date2)
        dates = np.array([date1, date2])
        LaterTimeProbe = np.max(dates)
        first_time_probe = np.min(dates)
        print(dates,LaterTimeProbe)
        last_session_date = (dates[(np.where(dates == LaterTimeProbe)[0])[0]])
        first_session_date = (dates[(np.where(dates == first_time_probe)[0])[0]])
        print(first_session_date)
        print(last_session_date)

        last_dt = datetime.strptime(str(last_session_date), "%Y%m%d")
        first_dt = datetime.strptime(str(first_session_date), "%Y%m%d")

        # Difference in days (absolute value so order doesn’t matter)
        day_delay = abs((last_dt - first_dt).days)

        print(f"Day delay: {day_delay}")
        print('---------')

        session_day_delay[nS] = day_delay

        sessSorted[0,(np.where(dates == LaterTimeProbe)[0])[0]] = 2
        print(sessSorted)
        s = nS * 2
        e = (nS+1) * 2
        #print(s,e)
        SortedSessionDates[s:e] = sessSorted
        print()

0
IMPORTING 2651 - Completed 2 sessions
[20231003 20231006] 20231006
20231003
20231006
Day delay: 3
---------
[[1 2]]

1
IMPORTING 2652 - Completed 2 sessions
[20231003 20231011] 20231011
20231003
20231011
Day delay: 8
---------
[[1 2]]

2
IMPORTING 2653 - Completed 2 sessions
[20231009 20231013] 20231013
20231009
20231013
Day delay: 4
---------
[[1 2]]

3
IMPORTING 2654 - Completed 2 sessions
[20231009 20231016] 20231016
20231009
20231016
Day delay: 7
---------
[[1 2]]

4
IMPORTING 2655 - Completed 2 sessions
[20231009 20231016] 20231016
20231009
20231016
Day delay: 7
---------
[[1 2]]

5
IMPORTING 2657 - Completed 2 sessions
[20231013 20231020] 20231020
20231013
20231020
Day delay: 7
---------
[[1 2]]

6
IMPORTING 2658 - Completed 2 sessions
[20231013 20231020] 20231020
20231013
20231020
Day delay: 7
---------
[[1 2]]

7
IMPORTING 2659 - Completed 2 sessions
[20231017 20231018] 20231018
20231017
20231018
Day delay: 1
---------
[[1 2]]

8
WARNING 2660-  is a single session..NOW FIXING

In [17]:
print(SortedSessionDates) # 1 == only 1 session was completed
print(len(SortedSessionDates))

[1. 2. 1. 2. 1. 2. 1. 2. 1. 2. 1. 2. 1. 2. 1. 2. 1. 0. 2. 1. 2. 1. 2. 1.
 2. 1. 2. 1. 2. 1. 2. 1. 1. 2. 1. 2. 2. 1. 1. 2. 1. 0. 2. 1. 1. 0. 2. 1.
 1. 2. 1. 2. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 2. 1. 1. 0. 1. 0. 1. 0. 1. 2.
 1. 2. 2. 1.]
76


In [18]:
# Clean_SortedSessionDates =  SortedSessionDates[~np.isnan(SortedSessionDates)] # old line 
# clean array from above to only consider sessiosns as 1st or 2nd, even those who only completed 1 session
Clean_SortedSessionDates = SortedSessionDates[SortedSessionDates != 0].astype(int) # New Line
print(len(Clean_SortedSessionDates))

print(session_day_delay)

# average_day_delay = np.nanmean(session_day_delay)

groups = np.full(session_day_delay.shape, np.nan)

# Assign group numbers
groups[(session_day_delay >= 0) & (session_day_delay <= 4)] = 1
groups[(session_day_delay > 4) & (session_day_delay <= 8)] = 2
groups[(session_day_delay > 8)] = 3

# groups[(session_day_delay >= 0) & (session_day_delay <= 6)] = 1
# groups[(session_day_delay > 6)] = 2

print(groups)
print(np.unique(groups, return_counts = True))

print(len(groups))

65
[ 3.  8.  4.  7.  7.  7.  7.  1. nan  1. 11.  1.  5.  4.  2.  4.  1. 38.
  5.  5. nan  2. nan  2. 11.  2. nan nan nan nan nan 13. nan nan nan 28.
  2. 28.]
[ 1.  2.  1.  2.  2.  2.  2.  1. nan  1.  3.  1.  2.  1.  1.  1.  1.  3.
  2.  2. nan  1. nan  1.  3.  1. nan nan nan nan nan  3. nan nan nan  3.
  1.  3.]
(array([ 1.,  2.,  3., nan]), array([13,  8,  6, 11]))
38


In [ ]:
# print(session_day_delay)
# print(np.nanmean(session_day_delay))
# print(np.nanstd(session_day_delay))


[ 3.  8.  4.  7.  7.  7.  7.  1. nan  1. 11.  1.  5.  4.  2.  4.  1. 38.
  5.  5. nan  2. nan  2. 11.  2. nan nan nan nan nan 13. nan nan nan 28.
  2. 28.]
7.7407407407407405
9.070067142520042


In [23]:
def MakeNewFolder(folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f'Folder {folder_name} created')
    else:
        print(f'Folder {folder_name} already exists')

In [24]:
DataDirOut = Path.home() / "Desktop" / "Visual Plasticty Pipeline" / "Outputs"
os.chdir(DataDirOut)

folder_name = 'SessionIndexKey'
MakeNewFolder(folder_name)

os.chdir(folder_name)

Folder SessionIndexKey already exists


In [25]:
dataOut = {
    'SessionDatesOrder': SortedSessionDates,
    'Clean_SessionDatesOrder': Clean_SortedSessionDates,
    'session_day_delay_groups' : groups
}

In [26]:
dnt = datetime.now() # add date and time bc im wreckless when saving ..
fdnt = dnt.strftime("%Y%m%d_%H%M%S") # set the above as a string ...

In [27]:
FileOutName = f'Clean_RCA_SessionOrder_Index_full'

In [28]:
ExporFileNom = f'{FileOutName}_{fdnt}.mat'
print(ExporFileNom)

Clean_RCA_SessionOrder_Index_full_20250922_123330.mat


In [29]:
# Export to .mat file
savemat(ExporFileNom, dataOut)

print(f"Data has been exported to '{ExporFileNom}'.")

Data has been exported to 'Clean_RCA_SessionOrder_Index_full_20250922_123330.mat'.
